#Import thư viện

In [ ]:
import random
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

#Load dữ liệu và build vocab :
Data sử dụng : **text8 corpus**

In [ ]:
!wget http://mattmahoney.net/dc/text8.zip

--2026-03-15 13:03:22--  http://mattmahoney.net/dc/text8.zip
Resolving mattmahoney.net (mattmahoney.net)... 20.119.76.151
Connecting to mattmahoney.net (mattmahoney.net)|20.119.76.151|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31344016 (30M) [application/zip]
Saving to: ‘text8.zip’

text8.zip           100%[===================>]  29.89M  25.3MB/s    in 1.2s    

2026-03-15 13:03:23 (25.3 MB/s) - ‘text8.zip’ saved [31344016/31344016]



In [ ]:
!unzip text8.zip

Archive:  text8.zip
  inflating: text8                   


In [ ]:
with open("text8", "r") as f:
    text = f.read()

print(type(text))
print(text[:200])

<class 'str'>
 anarchism originated as a term of abuse first used against early working class radicals including the diggers of the english revolution and the sans culottes of the french revolution whilst the term 


In [ ]:
# Chuẩn hóa str của corpus thành các token
tokens = text.strip().split()
print("Number of tokens:", len(tokens))
print(tokens[:20])

Number of tokens: 17005207
['anarchism', 'originated', 'as', 'a', 'term', 'of', 'abuse', 'first', 'used', 'against', 'early', 'working', 'class', 'radicals', 'including', 'the', 'diggers', 'of', 'the', 'english']


In [ ]:
# Chọn những từ xuất hiện >= 5 lần làm vocab
min_count = 5
counter = Counter(tokens)

vocab = [word for word, freq in counter.items() if freq >= min_count]
word2id = {word: i for i, word in enumerate(vocab)}
id2word = {i: word for word, i in word2id.items()}

corpus = [word2id[w] for w in tokens if w in word2id]

print("Vocab size:", len(vocab))
print("Corpus length:", len(corpus))

Vocab size: 71290
Corpus length: 16718844


In [ ]:
# Thực hiện cắt corpus để demo nhanh
corpus = corpus[:100000]

# Tạo postive pairs và tạo negative sampling distribution

In [ ]:
# Tạo postive pairs
def generate_pairs(corpus, window_size=2):
    pairs = []
    for i, target in enumerate(corpus):
        left = max(0, i - window_size)
        right = min(len(corpus), i + window_size + 1)
        for j in range(left, right):
            if i == j:
                continue
            context = corpus[j]
            pairs.append((target, context))
    return pairs

pairs = generate_pairs(corpus, window_size=2)
print("Number of pairs:", len(pairs))
print(pairs[:20])

Number of pairs: 399994
[(0, 1), (0, 2), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 3), (2, 4), (3, 1), (3, 2), (3, 4), (3, 5), (4, 2), (4, 3), (4, 5), (4, 6), (5, 3), (5, 4), (5, 6)]


In [ ]:
# Tạo Negative Sampling distribution
word_freq = np.array([counter[id2word[i]] for i in range(len(vocab))], dtype=np.float64)
neg_dist = word_freq ** 0.75
neg_dist = neg_dist / neg_dist.sum()

print(neg_dist[:20])

[3.24104708e-05 5.21975825e-05 3.08728692e-03 6.08680869e-03
 3.49511242e-04 9.54478249e-03 5.15803950e-05 9.86871965e-04
 8.26329465e-04 3.92691333e-04 4.52020500e-04 1.46813576e-04
 1.99232477e-04 1.57741825e-05 4.33934858e-04 1.47574545e-02
 4.98951548e-06 5.07442130e-04 1.34916449e-04 7.31838900e-03]


# Tạo class Dataset

In [ ]:
class Word2VecDataset(Dataset):
    def __init__(self, pairs, neg_dist, num_negatives, vocab_size):
        self.pairs = pairs
        self.neg_dist = neg_dist
        self.num_negatives = num_negatives
        self.vocab_size = vocab_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        target, context = self.pairs[idx]

        negatives = []
        while len(negatives) < self.num_negatives:
            neg = np.random.choice(self.vocab_size, p=self.neg_dist)
            if neg != context:
                negatives.append(neg)

        return (
            torch.tensor(target, dtype=torch.long),
            torch.tensor(context, dtype=torch.long),
            torch.tensor(negatives, dtype=torch.long)
        )

#Model SGNS bằng PyTorch

In [ ]:
class SGNS(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.target_embeddings = nn.Embedding(vocab_size, embed_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embed_dim)

        self.init_emb()

    def init_emb(self):
        init_range = 0.5 / self.target_embeddings.embedding_dim
        self.target_embeddings.weight.data.uniform_(-init_range, init_range)
        self.context_embeddings.weight.data.zero_()

    def forward(self, target_words, pos_context_words, neg_context_words):
        # target_words: [B]
        # pos_context_words: [B]
        # neg_context_words: [B, K]

        target_emb = self.target_embeddings(target_words)          # [B, D]
        pos_emb = self.context_embeddings(pos_context_words)       # [B, D]
        neg_emb = self.context_embeddings(neg_context_words)       # [B, K, D]

        # positive score
        pos_score = torch.sum(target_emb * pos_emb, dim=1)         # [B]
        pos_loss = F.logsigmoid(pos_score)                         # [B]

        # negative score
        neg_score = torch.bmm(neg_emb, target_emb.unsqueeze(2)).squeeze(2)  # [B, K]
        neg_loss = F.logsigmoid(-neg_score).sum(dim=1)             # [B]

        loss = -(pos_loss + neg_loss).mean()
        return loss

    def get_embeddings(self):
        return self.target_embeddings.weight.data + self.context_embeddings.weight.data

#Tạo DataLoader

In [ ]:
embed_dim = 100
num_negatives = 5
batch_size = 1024

dataset = Word2VecDataset(
    pairs=pairs,
    neg_dist=neg_dist,
    num_negatives=num_negatives,
    vocab_size=len(vocab)
)

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

#Train model

In [ ]:
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = SGNS(vocab_size=len(vocab), embed_dim=embed_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

epochs = 3

for epoch in range(epochs):
    total_loss = 0.0

    for target, pos_context, neg_context in tqdm(dataloader):
        target = target.to(device)
        pos_context = pos_context.to(device)
        neg_context = neg_context.to(device)

        optimizer.zero_grad()
        loss = model(target, pos_context, neg_context)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # if (step + 1) % 500 == 0:
        #     print(f"Epoch {epoch+1}, Step {step+1}/{len(dataloader)}, Loss: {total_loss/(step+1):.4f}")

    print(f"Epoch {epoch+1} done. Avg Loss: {total_loss/len(dataloader):.4f}")

Using device: cuda


100%|██████████| 391/391 [27:46<00:00,  4.26s/it]


Epoch 1 done. Avg Loss: 2.7764


100%|██████████| 391/391 [27:36<00:00,  4.24s/it]


Epoch 2 done. Avg Loss: 2.0780


100%|██████████| 391/391 [27:11<00:00,  4.17s/it]

Epoch 3 done. Avg Loss: 1.9410


#Đánh giá model

In [ ]:
embeddings = model.get_embeddings().cpu().numpy()
print(embeddings.shape)  # [vocab_size, embed_dim]

(71290, 100)


In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

def most_similar(word, embeddings, word2id, id2word, topn=10):
    if word not in word2id:
        return []

    idx = word2id[word]
    query_vec = embeddings[idx]

    sims = []
    for i in range(len(embeddings)):
        if i == idx:
            continue
        sim = cosine_similarity(query_vec, embeddings[i])
        sims.append((id2word[i], sim))

    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:topn]

def get_embedding(word) :
  idx = word2id[word]
  e = embeddings[idx]
  return e

def evalute(pairs, msg) :
  print(msg)
  for x, y in pairs :
    x_embedd = get_embedding(x)
    y_embedd = get_embedding(y)
    print(f"{x}, {y} : {cosine_similarity(x_embedd, y_embedd)}")
  print("--------")

In [ ]:
## TỪ nghĩa giống nhau
synonym_pairs = [
("big", "large"),
("small", "tiny"),
("quick", "fast"),
("smart", "intelligent"),
("car", "automobile"),
("begin", "start"),
("buy", "purchase"),
("child", "kid"),
("error", "mistake"),
("job", "occupation")
]

evaluate(synonym_pairs, "Synonyms")


related_pairs = [
("doctor", "hospital"),
("teacher", "school"),
("apple", "fruit"),
("dog", "animal"),
("car", "road"),
("phone", "call"),
("computer", "keyboard"),
("coffee", "cup"),
("music", "song"),
("king", "queen")
]

evaluate(related_pairs, "related words")


unrelated_pairs = [
("apple", "car"),
("dog", "banana"),
("computer", "tree"),
("city", "banana"),
("phone", "river"),
("music", "table"),
("king", "banana"),
("doctor", "mountain"),
("car", "banana"),
("teacher", "shoes")
]

evaluate(unrelated_pairs, "unrelated words")



Synonyms
big, large : 0.34100982546806335
small, tiny : -0.4982086718082428
quick, fast : 0.2563795745372772
smart, intelligent : -0.5751892924308777
car, automobile : -0.08618632704019547
begin, start : 0.1969752162694931
buy, purchase : 0.5993741750717163
child, kid : -0.7871618270874023
error, mistake : 0.15032227337360382
job, occupation : 0.042899440973997116
--------
related words
doctor, hospital : -0.10518211126327515
teacher, school : 0.6682468056678772
apple, fruit : -0.24022099375724792
dog, animal : 0.3379119634628296
car, road : 0.1010366678237915
phone, call : -0.6024764776229858
computer, keyboard : -0.04434693977236748
coffee, cup : 0.14916011691093445
music, song : 0.26838597655296326
king, queen : 0.5465137958526611
--------
unrelated words
apple, car : 0.2680679261684418
dog, banana : -0.3903975486755371
computer, tree : -0.1560603231191635
city, banana : -0.44502994418144226
phone, river : -0.679053783416748
music, table : 0.4470186233520508
king, banana : -0.495404